# 05 - Vector search with ChromaDB

Notebook 03 compared a question against five sentences with a for loop. Real
systems search millions of chunks, keep them on disk, and attach metadata to
every chunk. That is the job of a vector database. We use ChromaDB: it is
just a pip package, stores data in a local folder, and needs no server.

**What you will learn**

- How to create a ChromaDB collection that embeds automatically
- How to store chunks with metadata
- How to query and read distances
- How to filter results by metadata

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("../.env")
print("Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Key loaded: True


## Creating a collection

Two ChromaDB concepts:

- a **client** manages the database. PersistentClient saves everything to a
  folder, so the data survives after the notebook is closed. Notebook 06
  reuses this exact folder.
- a **collection** is like a table: a named set of chunks with their
  embeddings and metadata.

We hand the collection an embedding function so it calls OpenAI for us
automatically, both when storing chunks and when querying. We also set the
distance metric to cosine, which matches what we learned in notebook 03.

In [2]:
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path="../chroma_db")

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small",
)

collection = chroma_client.get_or_create_collection(
    name="aurora_docs",
    embedding_function=openai_ef,
    metadata={"hnsw:space": "cosine"},
)
print("Collection ready. Chunks currently stored:", collection.count())

Collection ready. Chunks currently stored: 0


## Loading and chunking the documents

The same chunking function we built in notebook 04:

In [3]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = end - overlap
    return chunks

ids, documents, metadatas = [], [], []

for path in sorted(Path("../data").glob("*.txt")):
    for i, chunk in enumerate(chunk_text(path.read_text())):
        ids.append(f"{path.stem}-{i}")
        documents.append(chunk)
        metadatas.append({"source": path.name})

print(f"Prepared {len(documents)} chunks from {len(set(m['source'] for m in metadatas))} files.")

Prepared 11 chunks from 6 files.


## Storing the chunks

One call stores everything. ChromaDB sends each chunk to the embedding
function and saves the text, the vector, and the metadata together. We use
upsert instead of add so the notebook can be re-run safely (existing ids get
overwritten instead of raising an error).

In [4]:
collection.upsert(ids=ids, documents=documents, metadatas=metadatas)
print("Chunks in collection:", collection.count())

Chunks in collection: 11


## Querying

Now the payoff. Ask in plain language, and ChromaDB embeds the question and
returns the closest chunks. One detail: it reports **distance**, not
similarity. With the cosine metric, distance = 1 - cosine similarity, so
**lower distance means a better match**.

In [5]:
results = collection.query(
    query_texts=["How many vacation days do employees get?"],
    n_results=3,
)

for doc, meta, dist in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0]
):
    print(f"distance {dist:.3f}  source: {meta['source']}")
    print(f"  {doc[:150]}...\n")

distance 0.523  source: 03-leave-policy.txt
  Aurora Dynamics - Employee Handbook: Leave Policy This policy applies to all full time employees of Aurora Dynamics. Annual leave: Every employee rece...

distance 0.551  source: 03-leave-policy.txt
  public holiday calendar. In addition, the whole company closes for a shared winter break from December 25 to January 1. Requesting leave: All leave is...

distance 0.602  source: 04-remote-work-policy.txt
  Aurora Dynamics - Employee Handbook: Remote Work Policy Aurora Dynamics uses a hybrid work model. Engineering and product teams work from the office o...



The leave policy chunks win, and the distances tell you how confident
the match is. Try a different topic:

In [6]:
results = collection.query(
    query_texts=["What should I do when a robot blinks orange?"],
    n_results=3,
)

for doc, meta, dist in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0]
):
    print(f"distance {dist:.3f}  source: {meta['source']}")
    print(f"  {doc[:150]}...\n")

distance 0.422  source: 05-support-runbook.txt
  Aurora Dynamics - Support Runbook: Common Carrier X2 Issues This runbook is used by the Lisbon support team to resolve frequent customer issues with t...

distance 0.613  source: 05-support-runbook.txt
  Reconnection takes about 2 minutes. If it fails, check that the hospital's integration gateway has power. Issue: compartment will not open after badge...

distance 0.689  source: 06-q1-2025-update.txt
  Aurora Dynamics - Q1 2025 Business Update (Internal) This is an internal summary shared with all employees in April 2025. Fleet growth: At the end of ...



## Filtering by metadata

Metadata is not just for display. The where argument restricts the search to
matching chunks, for example searching only inside one document. In real
systems this powers things like per-user permissions or date filters.

In [7]:
results = collection.query(
    query_texts=["battery"],
    n_results=2,
    where={"source": "02-product-guide.txt"},
)

for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"source: {meta['source']}")
    print(f"  {doc[:150]}...\n")

source: 02-product-guide.txt
  shift using the DockBrush station, which is sold separately. Pricing: hospitals do not buy the robot outright. Aurora Dynamics charges a subscription ...

source: 02-product-guide.txt
  Aurora Dynamics - Product Guide: Carrier X2 The Carrier X2 is the flagship delivery robot of Aurora Dynamics. Key specifications: - Payload capacity: ...



## Ollama alternative

Point the embedding function at Ollama instead of OpenAI. Use a different
collection name so the two sets of vectors never mix (vectors from different
models are not comparable):

In [8]:
# openai_ef = embedding_functions.OpenAIEmbeddingFunction(
#     api_key="ollama",
#     api_base="http://localhost:11434/v1",
#     model_name="nomic-embed-text",
# )
# collection = chroma_client.get_or_create_collection(
#     name="aurora_docs_ollama",
#     embedding_function=openai_ef,
#     metadata={"hnsw:space": "cosine"},
# )

## Where we stand

We now have a searchable knowledge base on disk (in the chroma_db folder at
the project root). Retrieval is solved. Notebook 06 connects it to the LLM
to complete the RAG loop.

## Exercise

1. Query for "how much does the robot cost" and check which source file the
   best chunk comes from.
2. Use a where filter to search for "leave" only in 04-remote-work-policy.txt.
   Compare with the unfiltered result.

In [9]:
# Try the exercise here
